# Custom PyTorch MLP for EC Classification

Architecture designed for small-data regime (512 training samples, ~50 EC classes):
- 2 hidden layers with batch normalization and dropout
- AdamW optimizer with weight decay
- Learning rate scheduling and early stopping
- Class-weighted loss to handle EC class imbalance

In [ ]:
from datasets import load_dataset

## loading from hugging face
dataset = load_dataset("tattabio/ec_classification")
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
print(train_df.shape)
print(test_df.shape)

In [ ]:
x_train_seq = train_df["Sequence"].tolist()
y_train_labels = train_df["Label"].tolist()

x_test_seq = test_df["Sequence"].tolist()
y_test_labels = test_df["Label"].tolist()

In [ ]:
!pip install dgeb

In [ ]:
import dgeb
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

embedding_model = dgeb.get_model("facebook/esm2_t6_8M_UR50D")

In [ ]:
## extract embeddings (last-token pooling, matching the rest of the project)
x_train_input1 = embedding_model.encode(x_train_seq)[:, -1, :]

In [ ]:
import numpy as np
x_test_p1 = embedding_model.encode(x_test_seq[:127])[:, -1, :]
last_test = embedding_model.encode(x_test_seq[127:])[:, -1, :]
x_test_input1 = np.vstack([x_test_p1, last_test])

## Encoding Functions
K-mer encoders for the 4 non-embedding inputs

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def kmerFunction(seq, k):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

## 2-mer
train_kmer2 = [kmerFunction(seq, 2) for seq in x_train_seq]
test_kmer2 = [kmerFunction(seq, 2) for seq in x_test_seq]

helper2 = CountVectorizer()
x_train_input2 = helper2.fit_transform(train_kmer2).toarray().astype(np.float32)
x_test_input2 = helper2.transform(test_kmer2).toarray().astype(np.float32)

weighted2 = TfidfVectorizer()
x_train_input3 = weighted2.fit_transform(train_kmer2).toarray().astype(np.float32)
x_test_input3 = weighted2.transform(test_kmer2).toarray().astype(np.float32)

## 3-mer
train_kmer3 = [kmerFunction(seq, 3) for seq in x_train_seq]
test_kmer3 = [kmerFunction(seq, 3) for seq in x_test_seq]

helper3 = CountVectorizer()
x_train_input4 = helper3.fit_transform(train_kmer3).toarray().astype(np.float32)
x_test_input4 = helper3.transform(test_kmer3).toarray().astype(np.float32)

weighted3 = TfidfVectorizer()
x_train_input5 = weighted3.fit_transform(train_kmer3).toarray().astype(np.float32)
x_test_input5 = weighted3.transform(test_kmer3).toarray().astype(np.float32)

print("input1 (ESM2):", x_train_input1.shape)
print("input2 (2-mer count):", x_train_input2.shape)
print("input3 (2-mer TF-IDF):", x_train_input3.shape)
print("input4 (3-mer count):", x_train_input4.shape)
print("input5 (3-mer TF-IDF):", x_train_input5.shape)

## Label Encoding
Convert string EC labels to integer indices for PyTorch

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
## fit on the union of train and test labels so all classes are seen
le.fit(y_train_labels + y_test_labels)
y_train_int = le.transform(y_train_labels)
y_test_int = le.transform(y_test_labels)
num_classes = len(le.classes_)
print(f"Number of EC classes: {num_classes}")

## Custom PyTorch MLP Architecture

**Design rationale:**
- **2 hidden layers** with sizes adapted to input dimension. Deep enough to learn non-linear interactions, shallow enough to not overfit on 512 samples.
- **Batch Normalization** stabilizes training and acts as light regularization. Critical for sparse k-mer features which have very different scales than dense ESM2 embeddings.
- **Dropout (0.4)** is aggressive — appropriate for our small dataset where overfitting is the main risk.
- **GELU activation** instead of ReLU. GELU is smoother and consistently performs slightly better in transformer-related tasks (which is where our embeddings come from).
- **Weight initialization** uses Kaiming/He init for proper gradient flow with GELU.
- **AdamW optimizer** decouples weight decay from gradient updates — better generalization than vanilla Adam.
- **Learning rate scheduling** with ReduceLROnPlateau prevents the model from getting stuck.
- **Early stopping** on validation loss prevents overfitting on small data.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)


class ProteinMLP(nn.Module):
    """Custom MLP for EC classification.

    Architecture: input -> [Linear -> BN -> GELU -> Dropout] x 2 -> Linear (logits)
    Hidden dims scale with input: max(input_dim/2, 128) for layer 1, half that for layer 2.
    This adapts capacity to feature dimension automatically across our 5 input types.
    """
    def __init__(self, input_dim, num_classes, dropout=0.4):
        super().__init__()
        ## adapt hidden sizes to input dimensionality
        # h1 = max(input_dim // 2, 128)
        # h2 = max(h1 // 2, 64)
        h1 = min(max(input_dim // 2, 128), 512)
        h2 = min(max(h1 // 2, 64), 256)

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h2, num_classes),
        )

        ## Kaiming initialization for better gradient flow with GELU
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

## Training Function with Early Stopping

**Design rationale:**
- We split off 15% of training data as a validation set to monitor for overfitting.
- **Class weights** in the loss handle EC class imbalance (some classes have many samples, others few).
- **Early stopping** with patience=15 lets the model train enough but stops when validation loss plateaus.
- **Gradient clipping** prevents exploding gradients on the high-dimensional sparse k-mer inputs.
- **Best model checkpointing** ensures we evaluate the model from its best epoch, not the final one.

In [ ]:
def train_and_evaluate(x_train, x_test, y_train, y_test, num_classes, encoding_name,
                       max_epochs=200, batch_size=32, lr=1e-3, weight_decay=1e-4, patience=15, dropout=0.4):
    """Train PyTorch MLP with early stopping, return test accuracy and macro F1."""

    ## train/val split (stratification disabled because rare classes can break it)
    try:
        x_tr, x_val, y_tr, y_val = train_test_split(
            x_train, y_train, test_size=0.15, random_state=42, stratify=y_train
        )
    except ValueError:
        x_tr, x_val, y_tr, y_val = train_test_split(
            x_train, y_train, test_size=0.15, random_state=42
        )

    ## convert to tensors
    x_tr_t = torch.tensor(x_tr, dtype=torch.float32).to(device)
    y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
    x_val_t = torch.tensor(x_val, dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)
    x_test_t = torch.tensor(x_test, dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(x_tr_t, y_tr_t), batch_size=batch_size, shuffle=True
    )

    ## class weights for imbalanced classes
    class_counts = Counter(y_tr)
    weights = np.ones(num_classes, dtype=np.float32)
    for c, count in class_counts.items():
        weights[c] = 1.0 / count
    weights = weights / weights.sum() * num_classes  # normalize
    weights_t = torch.tensor(weights, dtype=torch.float32).to(device)

    ## model, optimizer, loss, scheduler
    model = ProteinMLP(input_dim=x_train.shape[1], num_classes=num_classes).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )
    criterion = nn.CrossEntropyLoss(weight=weights_t)

    ## training loop with early stopping
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            ## gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        ## validation
        model.eval()
        with torch.no_grad():
            val_logits = model(x_val_t)
            val_loss = criterion(val_logits, y_val_t).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  [{encoding_name}] Early stopping at epoch {epoch+1}")
                break

    ## load best model and evaluate on test set
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(x_test_t)
        preds = test_logits.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='macro')
    print(f"  [{encoding_name}] accuracy={acc:.4f}, f1={f1:.4f}")
    return acc, f1

## Run on All 5 Encodings

In [ ]:
print("Training on ESM2 embeddings (last-token pooling)...")
acc1, f1_1 = train_and_evaluate(
    x_train_input1, x_test_input1, y_train_int, y_test_int, num_classes,
    encoding_name="ESM2"
)

In [ ]:
print("Training on 2-mer counts...")
acc2, f1_2 = train_and_evaluate(
    x_train_input2, x_test_input2, y_train_int, y_test_int, num_classes,
    encoding_name="2-mer count"
)

In [ ]:
print("Training on 2-mer TF-IDF...")
acc3, f1_3 = train_and_evaluate(
    x_train_input3, x_test_input3, y_train_int, y_test_int, num_classes,
    encoding_name="2-mer TF-IDF", dropout=0.2
)

In [ ]:
print("Training on 3-mer counts...")
acc4, f1_4 = train_and_evaluate(
    x_train_input4, x_test_input4, y_train_int, y_test_int, num_classes,
    encoding_name="3-mer count"
)

In [ ]:
print("Training on 3-mer TF-IDF...")
acc5, f1_5 = train_and_evaluate(
    x_train_input5, x_test_input5, y_train_int, y_test_int, num_classes,
    encoding_name="3-mer TF-IDF"
)

## Final Results Table

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Encoding": [
        "ESM2 embeddings (last-token)",
        "2-mer counts",
        "2-mer TF-IDF",
        "3-mer counts",
        "3-mer TF-IDF",
    ],
    "Accuracy": [acc1, acc2, acc3, acc4, acc5],
    "F1 (macro)": [f1_1, f1_2, f1_3, f1_4, f1_5],
})

results["Accuracy"] = results["Accuracy"].round(4)
results["F1 (macro)"] = results["F1 (macro)"].round(4)

print("Custom PyTorch MLP Results")
print("=" * 60)
print(results.to_string(index=False))